# เปรียบเทียบโมเดลจาก MLflow

อ่านผลการเทรนจาก `mlflow.db` (experiment `used-car-price`) แล้วตอบ 3 คำถาม:
1. โมเดลไหนดีที่สุดบน validation set และดีกว่า baseline แค่ไหน
2. ทำไมเลือก Ridge ที่ alpha = 1.0
3. ทำไมใช้ MAE เป็น optimizing metric ทั้งที่ MAPE ของ HistGradientBoosting ต่ำกว่า

ต้องรัน `python -m src.pipeline` และ `python -m src.retrain` ก่อน เพื่อให้มีไฟล์ `mlflow.db` ที่โฟลเดอร์หลักของโปรเจกต์

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import mlflow
import pandas as pd

warnings.filterwarnings("ignore")

# notebook อยู่ใน notebooks/ จึงถอยขึ้นหนึ่งระดับเพื่อหา mlflow.db
db = Path("..").resolve() / "mlflow.db"
print("พบ mlflow.db:", db.exists())

mlflow.set_tracking_uri(f"sqlite:///{db.as_posix()}")
runs = mlflow.search_runs(experiment_names=["used-car-price"])
runs = runs.rename(columns={"tags.mlflow.runName": "run_name"})

print("จำนวน run:", len(runs))
print([c for c in runs.columns if c.startswith("metrics.")])

## 1. ตารางเปรียบเทียบทุก run

In [ ]:
want = ["metrics.val_mae", "metrics.val_rmse", "metrics.val_mape",
        "metrics.test_mae", "metrics.test_mape"]
cols = ["run_name"] + [c for c in want if c in runs.columns]

table = runs[cols].rename(columns=lambda c: c.replace("metrics.", ""))
table.sort_values("val_mae").round(4)

ตารางนี้มี run `ridge_retrain` ซึ่งเทรนด้วย train + val จึงไม่มีค่า val (ว่าง) และมีเฉพาะค่า test

## 2. val MAE ของแต่ละโมเดล

In [ ]:
v = table.dropna(subset=["val_mae"]).sort_values("val_mae")

ax = v.plot.barh(x="run_name", y="val_mae", legend=False, figsize=(8, 4))
ax.invert_yaxis()
ax.set_xlabel("val MAE (INR)  -  lower is better")
ax.set_title("Validation MAE by run")
plt.tight_layout()
plt.show()

baseline (ใช้ราคากลางทายทุกคัน) มี MAE ประมาณ 415,937 ส่วนโมเดลที่ดีที่สุดคือ Ridge ประมาณ 120,615 ซึ่งผิดพลาดน้อยกว่า baseline ประมาณ 3.4 เท่า แสดงว่าโมเดลเรียนรู้ความสัมพันธ์ของสเปกรถกับราคาได้จริง

## 3. ทำไมเลือก alpha = 1.0

In [ ]:
r = v[v["run_name"].str.startswith("ridge_alpha=")].copy()
r["alpha"] = r["run_name"].str.extract(r"=([\d.]+)")[0].astype(float)
r = r.sort_values("alpha")

plt.figure(figsize=(7, 4))
plt.plot(r["alpha"], r["val_mae"], marker="o")
plt.xscale("log")
plt.xlabel("alpha (log scale)")
plt.ylabel("val MAE (INR)")
plt.title("Ridge: val MAE vs alpha")
plt.show()

r[["alpha", "val_mae"]].round(0)

alpha ควบคุมความแรงของการลดน้ำหนักสัมประสิทธิ์ (regularization)

- alpha น้อยเกินไป (0.1) โมเดลปรับตามข้อมูลเทรนมาก ผิดพลาดบน val สูงกว่าเล็กน้อย
- alpha มากเกินไป (10, 50, 100) โมเดลเรียบเกินไป ผิดพลาดเพิ่มขึ้นเรื่อยๆ
- alpha = 1.0 ให้ val MAE ต่ำสุด จึงถูกเลือก

เลือกจาก **val set** ไม่ใช่ test set เพื่อให้ test ยังเป็นข้อมูลที่โมเดลไม่เคยถูกใช้ตัดสินใจ

## 4. MAE กับ MAPE ให้ภาพไม่ตรงกัน

In [ ]:
if "val_mape" in v.columns:
    m = v[~v["run_name"].str.startswith("ridge_alpha=") | (v["run_name"] == "ridge_alpha=1.0")]
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    m.plot.barh(x="run_name", y="val_mae", ax=ax[0], legend=False, title="val MAE (INR)")
    m.plot.barh(x="run_name", y="val_mape", ax=ax[1], legend=False, title="val MAPE (fraction)")
    for a in ax:
        a.invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print("ไม่พบ val_mape ใน MLflow ให้ดูค่าจากตารางใน README")

จากตาราง validation ใน README:

| โมเดล | MAE | RMSE | MAPE |
|---|---|---|---|
| Ridge (alpha=1.0) | 120,615 | 187,566 | 17.1% |
| HistGradientBoosting | 129,150 | 356,127 | 14.4% |
| RandomForest | 135,776 | 363,139 | 15.0% |

สองโมเดลต้นไม้ (HistGradientBoosting, RandomForest) มี MAPE ต่ำกว่า Ridge แต่ MAE สูงกว่า และ RMSE สูงกว่ามาก (ประมาณ 1.9 เท่า) ซึ่งสอดคล้องกับ:

- **MAPE** วัดความผิดพลาดเป็นสัดส่วนของราคาจริง ให้น้ำหนักรถราคาถูกมาก (พลาด 50,000 บนรถ 200,000 = 25%)
- **MAE** วัดความผิดพลาดเป็นเงินรูปีตรงๆ ให้น้ำหนักทุกคันเท่ากัน
- **RMSE** ยกกำลังสองก่อนเฉลี่ย จึงไวต่อการพลาดครั้งใหญ่ ค่าที่สูงมากแปลว่าโมเดลต้นไม้มีบางคันที่พลาดหนัก (น่าจะเป็นรถราคาสูง แต่ต้องเช็กกับข้อมูลรายคันก่อนสรุป)

ดังนั้นโมเดลต้นไม้แม่นกว่าในเชิงสัดส่วนกับรถทั่วไป แต่พลาดหนักกว่าในรถบางกลุ่ม ส่วน Ridge พลาดน้อยกว่าในหน่วยเงินเฉลี่ย

**เหตุผลที่ทีมเลือก MAE เป็น optimizing metric:** ผู้ขายสนใจว่าราคาที่แนะนำคลาดเคลื่อนกี่บาทโดยเฉลี่ย และ MAE ไม่ถูกดึงโดยรถไม่กี่คันมากเท่า RMSE ส่วน MAPE ถูกใช้เป็น **gating metric** (ต้อง ≤ 20% บน test) เพื่อกันกรณีที่ผิดพลาดสัมพัทธ์สูงเกินรับได้ การใช้สองตัวคู่กันครอบคลุมทั้งมุมเงินและมุมสัดส่วน

ข้อควรระวัง: การที่ Ridge ชนะด้วย MAE ไม่ได้แปลว่าดีที่สุดทุกมุม ถ้าเปลี่ยน metric หลักเป็น MAPE ผลการเลือกโมเดลอาจเปลี่ยน

## 5. champion v1 กับ v2 บน test set

ผลจาก `python -m src.retrain` (test set ปี ≥ 2018 ชุดเดียวกัน):

| เวอร์ชัน | ข้อมูลเทรน | test MAE | test MAPE |
|---|---|---|---|
| v1 | train (≤ 2016) | 184,906 | 19.6% |
| v2 | train + val (≤ 2017) | 177,922 | 19.2% |

v2 ดีขึ้นเพราะได้ข้อมูลปี 2017 เพิ่ม ซึ่งใกล้ช่วงเวลาของ test มากกว่า (ไม่ได้เปลี่ยนโมเดลหรือ alpha) แต่ test MAPE ของทั้งสองเวอร์ชัน (19.2-19.6%) ใกล้เกณฑ์ 20% มาก และสูงกว่า val MAPE (17.1%) ซึ่งสอดคล้องกับที่ระบบตรวจพบ drift เมื่อข้อมูลใหม่ขึ้น